In [1]:
import pandas as pd
import requests
from dotenv import load_dotenv
import os
from pathlib import Path
import difflib
from collections import defaultdict

In [2]:
load_dotenv()

True

In [3]:
DATA_DIR = Path.cwd().parent / "Data"

In [4]:
df = pd.read_csv(DATA_DIR / "atp_tennis.csv")

In [5]:
API_KEY = os.environ["ODDS_API_KEY"]

In [6]:
base = "https://api.the-odds-api.com/v4"
all_events = requests.get(f"{base}/sports", params={"apiKey" : API_KEY,
                                                    "all" : "true"}).json()

tennis_events = [event for event in all_events if event["group"] == "Tennis" and "atp" in event["key"]]

In [7]:
cutoff = pd.Timestamp.today() - pd.DateOffset(months=13)
df["Date"] = pd.to_datetime(df["Date"])
tourn_data = df.sort_values("Date").groupby("Tournament")[["Series", "Surface", "Court", "Date"]].last().reset_index()
recent = tourn_data[tourn_data["Date"] >= cutoff][["Tournament", "Series", "Surface", "Court", "Date"]]

In [8]:
keys = [s["key"] for s in tennis_events]

In [9]:
matches = difflib.get_close_matches(keys[0], list(recent["Tournament"]), n=1, cutoff=0)

In [10]:
lookup = defaultdict(list)
for event in keys:
    
    lookup[event] = "".join(difflib.get_close_matches(event, list(recent["Tournament"]), n=1, cutoff=0))

In [11]:
len(lookup)

23

In [12]:
lookup

defaultdict(list,
            {'tennis_atp_aus_open_singles': 'Dubai Tennis Championships',
             'tennis_atp_barcelona_open': 'Barcelona Open',
             'tennis_atp_canadian_open': 'Canadian Open',
             'tennis_atp_china_open': 'Argentina Open',
             'tennis_atp_cincinnati_open': 'Brisbane International',
             'tennis_atp_dubai': 'Dubai Tennis Championships',
             'tennis_atp_french_open': 'French Open',
             'tennis_atp_halle_open': 'Halle Open',
             'tennis_atp_hamburg_open': 'Hamburg Open',
             'tennis_atp_indian_wells': 'Dubai Tennis Championships',
             'tennis_atp_italian_open': 'Australian Open',
             'tennis_atp_japan_open': 'Australian Open',
             'tennis_atp_madrid_open': 'Mutua Madrid Open',
             'tennis_atp_miami_open': 'Argentina Open',
             'tennis_atp_monte_carlo_masters': 'Monte Carlo Masters',
             'tennis_atp_munich': 'Vienna Open',
             'tenni

In [13]:
changes = {'tennis_atp_aus_open_singles' : 'Australian Open',
           'tennis_atp_china_open' : 'China Open',
           'tennis_atp_cincinnati_open' : 'Western & Southern Financial Group Masters',
           'tennis_atp_indian_wells' : 'BNP Paribas Open',
           'tennis_atp_italian_open' : 'Internazionali BNL d\'Italia',
           'tennis_atp_japan_open' : 'Japan Open Tennis Championships',
           'tennis_atp_tennis_miami_open' : 'Miami Open',
           'tennis_atp_munich' : 'BMW Open',
           'tennis_atp_paris' : 'BNP Paribas Masters',
           'tennis_atp_qatar_open' : 'Qatar Exxon Mobil Open',
           'tennis_atp_us_open' : 'US Open',
           'tennis_atp_washington_open' : 'Citi Open'
           }
lookup.update(changes)

In [14]:
string = 'Cincinatti Open'
string in recent["Tournament"].values

False

In [15]:
lookup_df = pd.DataFrame(lookup.items())
lookup_df = lookup_df.rename(columns={0 : "Odds_Key", 1 : "Dataset_Name"})

In [16]:
surface_list = []

for tournament_name in lookup_df["Dataset_Name"]:
    if not recent.loc[recent["Tournament"] == tournament_name, ["Tournament", "Surface"]]["Surface"].empty:
        surface_list.append(recent.loc[recent["Tournament"] == tournament_name, ["Tournament", "Surface"]]["Surface"].item())
    else:
        surface_list.append("No Information")
    
print(surface_list)

['Hard', 'Clay', 'Hard', 'Hard', 'Hard', 'Hard', 'Clay', 'Grass', 'Clay', 'Hard', 'Clay', 'Hard', 'Clay', 'Clay', 'Clay', 'Clay', 'Clay', 'Hard', 'Grass', 'Hard', 'Hard', 'Hard', 'Grass', 'Hard', 'Hard']


In [17]:
surface_series = pd.Series(surface_list)
lookup_df = pd.concat([lookup_df, surface_series], axis=1)

In [18]:
lookup_df

,Odds_Key,Dataset_Name,0
0,tennis_atp_aus_open_singles,Australian Open,Hard
1,tennis_atp_barcelona_open,Barcelona Open,Clay
2,tennis_atp_canadian_open,Canadian Open,Hard
3,tennis_atp_china_open,China Open,Hard
4,tennis_atp_cincinnati_open,Western & Southern Financial Group Masters,Hard
5,tennis_atp_dubai,Dubai Tennis Championships,Hard
6,tennis_atp_french_open,French Open,Clay
7,tennis_atp_halle_open,Halle Open,Grass
8,tennis_atp_hamburg_open,Hamburg Open,Clay
9,tennis_atp_indian_wells,BNP Paribas Open,Hard


In [19]:
lookup_df = lookup_df.rename(columns={0 : "Surface"})
lookup_df.to_csv('../Data/lookup_table.csv', index=False)